# Create image stacks with only caax and cell channels
### Purpose: reduces file size during subsequent file processing and prevents the actin channel from being a source of bias during analysis

In [ ]:


import bioio_ome_tiff
import bioio_tifffile


In [ ]:
input_dirpath = Path(input())

In [ ]:
# Obtain list of images to process from "selected_imgs.csv"
img_list_path = utils.get_proc_dirpath(input_dirpath) / dn.tables_dirname / 'selected_imgs.csv'
assert img_list_path.is_file()
img_list_df = pd.read_csv(img_list_path)
cols_to_use = ['image name', 'wellID', 'scene', 'img idx', 'experiment', 'DIV', 'ch0', 'ch1', 'caax ch', 'cell ch']
img_list_df = img_list_df[cols_to_use]
img_list_df.head()

In [ ]:
# Create output directories

# Main directory for 'caax_cell_stack'
proc_dirpath = utils.get_proc_dirpath(input_dirpath)
output_dirname = 'caax_cell_stack'
output_dirpath = proc_dirpath / output_dirname

# Directory for cells with 2 caax channels - stores the second caax-cell stack
dup_dirpath = output_dirpath / 'caax_cell_stack_othercaax'
dup_dirpath.mkdir(parents=True, exist_ok=True)

In [ ]:
# Stores info about channels and CAAX fluorescent proteins
caaxch_d = {0:'mRubycaax', 1:'GFPcaax'}

In [ ]:
# create lists to store info about proceessed images
wellIDs = []
scenes = []
stackimgnames = []
origimgnames = []
caaxchs = []
cellchs = []
dups = []
outputdirnames = []

# Create stacks with CAAX and cell ch
num_imgs = len(img_list_df)

for i, row in tqdm(img_list_df.iterrows()):
    imgname = row['image name']
    imgpath = input_dirpath / row['image name']
    
    # open image
    if imgpath.is_file():
        print(f'Processing img {i}/{num_imgs}: {imgname}')
        img_file = BioImage(imgpath, reader=bioio_ome_tiff.Reader)
        img = img_file.data
        
    cellch = row['cell ch']
    
    for key, val in caaxch_d.items():
        
        # split files into folders depending on whether the caax channel is marked for analysis
        if row[f'ch{key}']=='caax':
            savename = imgname.split('.ome.tif')[0] + f'_{val}.ome.tif'

            if row['caax ch'] == key:
                stackimgpath = output_dirpath / savename
                dups.append(False)
                save_dirpath = output_dirpath
                outputdirnames.append(save_dirpath.name)
            else:
                stackimgpath = dup_dirpath / savename
                dups.append(True)
                save_dirpath = dup_dirpath
                outputdirnames.append(save_dirpath.name)
                  
            ch_subset = [int(key), int(cellch)]
            utils.create_ch_subset(ch_subset, stackimgpath, img=img, img_file=img_file, output_dir=save_dirpath)
            
            stackimgnames.append(stackimgpath.name)
            origimgnames.append(imgpath.name)
            wellIDs.append(row['wellID'])
            scenes.append(row['scene'])
            caaxchs.append(key)
            cellchs.append(cellch)
                
print('Done making channel subsets!')
            
df = pd.DataFrame({'wellID': wellIDs, 'scene': scenes,
                   'output dirname': outputdirnames, 'output image name': stackimgnames, 
                   'output caax ch': 0, 'output cell ch': 1, 
                   'input dirname': input_dirpath.name, 'input img name': origimgnames, 
                   'input caax ch': caaxchs, 'input cell ch': cellchs, 'duplicate CAAX': dups})

# Save info about caax cell stack processing
output_df_name = 'caax_cell_stack_list.csv'
output_df_path = img_list_path.parent / output_df_name
df.to_csv(output_df_path, index=False)

df.head()